In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rosbags
from pathlib import Path
import importlib

import ekf.class_ekf
importlib.reload(ekf.class_ekf)
from ekf.class_ekf import EKF

import data.ford_data_adapter
importlib.reload(data.ford_data_adapter)
from data.ford_data_adapter import FordDataAdapter

In [ ]:
adapter = FordDataAdapter(Path("2017-10-26-V2-Log6-sorted.bag"))
print(adapter.bag_path.resolve())

In [ ]:
initial_pose_set = False

ekf = EKF()

estimations_x = []
estimations_y = []
ground_truth_x = []
ground_truth_y = []

for i, (action, value, timestamp) in enumerate(adapter):
    if action == 'predict':
        if initial_pose_set:
            ekf.predict(value)
    elif action == 'update':
        if not initial_pose_set:
            ekf.set_pose(value)
            initial_pose_set = True
        else:
            ekf.update(value)
        estimations_x.append(ekf.x[0])
        estimations_y.append(ekf.x[1])
    elif action == 'ground_truth':
        ground_truth_x.append(value[0])
        ground_truth_y.append(value[1])

    if i > 30000:
        break

In [ ]:
plt.figure(figsize=(10, 8))
plt.plot(ground_truth_x, ground_truth_y, 'g-', linewidth=5, label='Ground Truth')
plt.plot(estimations_x, estimations_y, 'b-', linewidth=2, label='EKF')
plt.xlabel('X (м)')
plt.ylabel('Y (м)')
plt.title('Сравнение EKF с эталонной траекторией')
plt.legend()
plt.grid(True)
plt.axis('equal')

plt.savefig(Path("results") / "comparison_2017-10-26-V2-Log6.png", dpi=150, bbox_inches='tight')
plt.show()
plt.close()